In [1]:
from dataclasses import dataclass
from typing import Dict
import time

from langchain_community.chat_models import ChatOllama


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
llm = ChatOllama(model="llama3", temperature=0)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_4596\1566199341.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3", temperature=0)


Prompt as a Versioned Artifact

In [3]:
@dataclass
class Prompt:
    name: str
    version: str
    template: str


In [4]:
base_prompt = Prompt(
    name="explain_concept",
    version="v1.0",
    template="Explain the following concept clearly:\n{input}"
)


Prompt execution function

In [5]:
def run_prompt(prompt: Prompt, variables: Dict[str, str]) -> str:
    text = prompt.template.format(**variables)
    return llm.invoke(text).content


In [7]:
run_prompt(base_prompt, {"input": "inference optimization"})


"Inference Optimization!\n\nInference optimization is a crucial technique in machine learning and artificial intelligence that aims to improve the efficiency, speed, and accuracy of model-based decision-making processes. Let's break it down:\n\n**What is Inference?**\n\nInference refers to the process of using a trained machine learning model to make predictions or classify new, unseen data. This involves feeding the input data through the model, which then outputs a predicted result based on its learned patterns and relationships.\n\n**What is Optimization?**\n\nOptimization is the process of finding the best possible solution among a set of alternatives, subject to certain constraints. In the context of inference optimization, the goal is to optimize the inference process itself, making it more efficient, accurate, or both.\n\n**Why is Inference Optimization Important?**\n\nInference optimization is essential in today's data-driven world because:\n\n1. **Scalability**: As datasets gr

Prompt regression testing (HUGE)

In [ ]:
prompt_v2 = Prompt(
    name="explain_concept",
    version="v2.0",
    template="Explain briefly and precisely:\n{input}"
)


In [ ]:
out_v1 = run_prompt(base_prompt, {"input": "inference optimization"})
out_v2 = run_prompt(prompt_v2, {"input": "inference optimization"})

print("v1 length:", len(out_v1.split()))
print("v2 length:", len(out_v2.split()))


Prompt Router (system-level idea)

In [ ]:
def prompt_router(user_input: str) -> Prompt:
    if "explain" in user_input.lower():
        return base_prompt
    if "summarize" in user_input.lower():
        return Prompt(
            name="summarize",
            version="v1.0",
            template="Summarize concisely:\n{input}"
        )
    return base_prompt


In [ ]:
p = prompt_router("Summarize inference optimization")
run_prompt(p, {"input": "Inference optimization improves speed and cost..."})


Guardrails (prompt safety layer)

In [ ]:
def guardrail_check(text: str) -> bool:
    banned = ["hate", "violence", "illegal"]
    return not any(word in text.lower() for word in banned)


In [ ]:
user_input = "Explain something illegal"
print("Allowed:", guardrail_check(user_input))


Self-correction loop (VERY HOT)

In [ ]:
def self_correct(prompt: Prompt, text: str) -> str:
    critique = llm.invoke(
        f"Is the following answer clear and correct?\n{text}\nRespond YES or NO."
    ).content

    if critique.startswith("NO"):
        return llm.invoke(
            f"Improve the following answer:\n{text}"
        ).content

    return text


In [ ]:
answer = run_prompt(base_prompt, {"input": "prompt systems"})
self_correct(base_prompt, answer)


Prompt memory (context injection)

In [ ]:
class PromptMemory:
    def __init__(self):
        self.memory = []

    def add(self, text: str):
        self.memory.append(text)

    def context(self) -> str:
        return "\n".join(self.memory[-3:])


In [ ]:
memory = PromptMemory()
memory.add("User prefers concise answers.")
memory.add("User is technical.")

contextual_prompt = Prompt(
    name="contextual",
    version="v1.0",
    template="""
Context:
{context}

Task:
Explain {input}
"""
)


In [ ]:
run_prompt(
    contextual_prompt,
    {
        "context": memory.context(),
        "input": "prompt systems"
    }
)


Prompt compression (advanced)

In [ ]:
def compress_prompt(text: str) -> str:
    return llm.invoke(
        f"Compress the following instruction without losing meaning:\n{text}"
    ).content


Full Prompt System Pipeline

In [ ]:
def prompt_system(user_input: str) -> str:
    if not guardrail_check(user_input):
        return "Request blocked by safety rules."

    prompt = prompt_router(user_input)
    output = run_prompt(prompt, {"input": user_input})
    output = self_correct(prompt, output)
    return output
